# 04 — Export Final Model

Build an `EModelExportFinalModelScanConfig`, expand it via `GridScanGenerationTask`,
and run the `EModelExportFinalModelTask` for each coordinate. The task seeds its
working directory from the analysis output, then calls `export_emodels_hoc` and/or
`export_emodels_sonata` with the seeds the user picked.

**Reads from:** `obi-output/03_analysis_and_validation/grid_scan/0/`.

**Writes to:** `obi-output/04_export_final_model/grid_scan/0/` — the working directory
plus the new `export_emodels_hoc/` (one `.hoc` per validated model) and
`export_emodels_sonata/` (with `nodes.h5`).


## Imports

In [1]:
from pathlib import Path

import obi_one as obi
from obi_one.scientific.tasks.emodel_optimization._04_export_final_model.blocks import (
    ExportInitialize,
    ExportSettings,
)


## Build the scan config

In [2]:
previous_stage = (
    Path.cwd() / "../../../obi-output/03_analysis_and_validation/grid_scan/0"
).resolve()
assert previous_stage.exists(), (
    f"{previous_stage} not found — run 03_analysis_and_validation.ipynb first."
)

scan_config = obi.EModelExportFinalModelScanConfig(
    initialize=ExportInitialize(
        previous_stage_output_path=previous_stage,
        emodel="L5PC",
        species="rat",
        brain_region="SSCX",
    ),
    export_settings=ExportSettings(
        export_hoc=True,
        export_sonata=True,
        only_validated=False,
        only_best=False,
        seeds=(1,),  # must be a subset of the seeds run in stage 02
    ),
)


## Run the grid scan

In [3]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/04_export_final_model/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)


[2026-05-05 14:35:47,830] INFO: Seeded /Users/james/Documents/obi/code/obi-main/obi-output/03_analysis_and_validation/grid_scan/0/config -> /Users/james/Documents/obi/code/obi-main/obi-output/04_export_final_model/grid_scan/0/config
[2026-05-05 14:35:47,831] INFO: Seeded /Users/james/Documents/obi/code/obi-main/obi-output/03_analysis_and_validation/grid_scan/0/morphologies -> /Users/james/Documents/obi/code/obi-main/obi-output/04_export_final_model/grid_scan/0/morphologies
[2026-05-05 14:35:47,833] INFO: Seeded /Users/james/Documents/obi/code/obi-main/obi-output/03_analysis_and_validation/grid_scan/0/mechanisms -> /Users/james/Documents/obi/code/obi-main/obi-output/04_export_final_model/grid_scan/0/mechanisms
[2026-05-05 14:35:47,907] INFO: Seeded /Users/james/Documents/obi/code/obi-main/obi-output/03_analysis_and_validation/grid_scan/0/ephys_data -> /Users/james/Documents/obi/code/obi-main/obi-output/04_export_final_model/grid_scan/0/ephys_data
[2026-05-05 14:35:47,908] INFO: Seeded /

--No graphics will be displayed.


[2026-05-05 14:35:48,401] WARNING: Exporting a model that did not pass validation.
[2026-05-05 14:35:48,597] INFO: In compute_responses, 1 emodels found to evaluate.
[2026-05-05 14:36:41,459] WARNING: Exporting a model that did not pass validation.


[PosixPath('/Users/james/Documents/obi/code/obi-main/obi-output/04_export_final_model/grid_scan/0')]

## Inspect the exported model

In [4]:
coord_root = Path(grid_scan.single_configs[0].coordinate_output_root).resolve()
print("Coordinate output:", coord_root)

for tree in ("export_emodels_hoc", "export_emodels_sonata"):
    print(f"\n{tree}/")
    for p in sorted((coord_root / tree).rglob("*"))[:20]:
        if p.is_file():
            print(" ", p.relative_to(coord_root))


Coordinate output: /Users/james/Documents/obi/code/obi-main/obi-output/04_export_final_model/grid_scan/0

export_emodels_hoc/
  export_emodels_hoc/emodel=L5PC__species=rat__brain_region=SSCX__seed=1/C060114A5.asc
  export_emodels_hoc/emodel=L5PC__species=rat__brain_region=SSCX__seed=1/model.hoc

export_emodels_sonata/
  export_emodels_sonata/emodel=L5PC__species=rat__brain_region=SSCX__seed=1/C060114A5.asc
  export_emodels_sonata/emodel=L5PC__species=rat__brain_region=SSCX__seed=1/model.hoc
  export_emodels_sonata/emodel=L5PC__species=rat__brain_region=SSCX__seed=1/nodes.h5
